# 🏗️ Notebook 1: Google Calendar — Requirements & Architecture

## 🛠️ Setup

```bash
cd 06-system-designs/google-calendar
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## What we're designing

A calendar service like Google Calendar. Users:

- create **events** (one-off or recurring — "every Monday"),
- invite **guests** who can RSVP (yes/no/maybe),
- book **meeting rooms** as resources,
- share calendars with colleagues (read-only or read/write),
- get **reminders** by push or email.

Two parts are genuinely hard and deserve their own notebooks:

1. **Recurring events** — one "every weekday at 9am forever" event should NOT create infinite rows.
2. **Timezones** — storing "3pm" is ambiguous. Is that Tokyo 3pm or New York 3pm? And does daylight-saving time shift it?


## Functional requirements

| Area | What the user can do |
|---|---|
| Events | Create, edit, delete one-off or recurring events |
| Invites | Invite guests, see their RSVP (yes/no/maybe) |
| Rooms | Book a meeting room (a "resource" with its own calendar) |
| Availability | "Find me 30 min when Alice, Bob, and Room 7 are all free" |
| Sharing | Share my calendar: free/busy only, read, or read/write |
| Reminders | Get a push/email 10 minutes before an event |

## Non-functional requirements

- **Correctness of time**: handle timezones and daylight-saving transitions.
- **Low read latency**: opening the day view should feel instant (<200 ms).
- **Durable reminders**: a reminder must fire even if a server restarts.
- **Eventual consistency** across regions is acceptable for calendar views; invitations and RSVPs should converge within seconds.


## Back-of-envelope sizing

Rough numbers — the point is to reason about *orders of magnitude*, not to be exact.
Rather than assert the numbers, let's compute them, because two of them are surprising:
the read:write ratio is smaller than people guess, and the reminder peak is much larger.

Assumptions (change them and rerun):

- **1 B** active users.
- Each user **creates ~3 events/week** and **opens the calendar ~12×/day**.
- The average user has **~2 events per day** that carry a reminder.
- An event row is **~1 KB** (title, times, rrule, room, ACL refs — the body lives elsewhere).

In [ ]:
# Calendar sizing. Every figure below is derived, so you can see which input drives it.
users            = 1_000_000_000
creates_per_week = 3            # events a user creates
opens_per_day    = 12           # calendar/app opens (each fetches a week view)
event_bytes      = 1_000
years_retained   = 10
reminders_per_user_per_day = 2
peak_ratio       = 3            # daily peak / daily average

SEC_PER_DAY, SEC_PER_WEEK = 86_400, 604_800

# ---- traffic ----
writes_s = users * creates_per_week / SEC_PER_WEEK
reads_s  = users * opens_per_day / SEC_PER_DAY
print(f"avg write QPS       : {writes_s:>12,.0f} /s")
print(f"avg read  QPS       : {reads_s:>12,.0f} /s")
print(f"read:write ratio    : {reads_s/writes_s:>12,.0f} : 1")
print(f"peak read QPS       : {reads_s*peak_ratio:>12,.0f} /s  ({peak_ratio}x)")

# ---- storage ----
events_total = users * creates_per_week * 52 * years_retained
storage_pb   = events_total * event_bytes / 1e15
print(f"\nevents over {years_retained} yrs   : {events_total/1e12:>12,.1f} trillion")
print(f"event-row storage   : {storage_pb:>12,.1f} PB   (+ replication ×3 = "
      f"{storage_pb*3:,.0f} PB)")

# ---- the number that actually hurts: reminder fan-out ----
# Reminders are not smooth. Humans schedule on the hour, so most of a day's reminders
# fire inside 24 one-minute windows.
reminders_day  = users * reminders_per_user_per_day
on_the_hour    = 0.60                      # fraction landing on an :00 boundary
avg_reminder_s = reminders_day / SEC_PER_DAY
peak_reminder_s = reminders_day * on_the_hour / (24 * 60)
print(f"\nreminders/day       : {reminders_day/1e9:>12,.1f} B")
print(f"avg reminder rate   : {avg_reminder_s:>12,.0f} /s")
print(f"PEAK reminder rate  : {peak_reminder_s:>12,.0f} /s  "
      f"({peak_reminder_s/avg_reminder_s:.0f}x the average)")

### What the numbers actually say

- **Reads beat writes by ~30:1, not 1000:1.** Calendars are read-heavy but not
  Twitter-heavy. That is comfortably cacheable; the read path is not the hard part.
- **Storage is tens of petabytes** — large, but this is boring relational data.
  Sharding by `user_id` (or `calendar_id`) handles it. Storage is not the hard part
  either.
- **Reminders are the spike.** Nobody schedules a meeting for 10:37. Sixty percent of
  reminders fire inside 24 one-minute windows a day, so the peak is roughly **36× the
  average**. Provisioning the reminder path for the average rate guarantees you miss
  reminders every hour, on the hour. That is why the scheduler is a separate durable
  timer queue with its own capacity plan — see the `reminder-alert` lab.

**Now the counter-example that makes idea #1 concrete.** The storage figure above assumed
we store one row per *created event*. Watch what happens if we instead materialize every
future occurrence of a recurring event.

In [ ]:
# Store the RULE vs. store every OCCURRENCE -- for one plausible mix of events.
recurring_share = 0.30      # ~30% of created events repeat
horizon_years   = 5         # how far ahead you'd have to materialize a "no end date" rule
occ_per_year    = 250       # a weekday standup

created  = users * creates_per_week * 52 * years_retained
recurring = created * recurring_share
one_off   = created - recurring

rows_rule_based  = created                                    # 1 row each, recurring or not
rows_materialized = one_off + recurring * occ_per_year * horizon_years

for label, rows in [("store the rule    ", rows_rule_based),
                    ("materialize rows  ", rows_materialized)]:
    print(f"{label}: {rows/1e12:>8,.1f} T rows   {rows*event_bytes/1e15:>8,.0f} PB")

print(f"\nmaterializing costs {rows_materialized/rows_rule_based:,.0f}x the rows "
      f"and {(rows_materialized-rows_rule_based)*event_bytes/1e15:,.0f} PB extra.")
print("""
And the storage is the cheap part of that bill. The real costs are:
  * "move my standup to 9:30" becomes an UPDATE over ~1,250 rows, in one transaction
  * the series has no end, so a cron must keep extending the horizon forever
  * every edit races that cron
The rule-based version pays instead at READ time: every day-view has to expand rules.
That is a good trade -- reads are cheap, bounded (one window), and cacheable -- but it
is a trade, not a free win. It is also why the query path needs an index on
(calendar_id, starts_at) *plus* a separate scan of open-ended rules.""")

## High-level architecture

```
                     ┌──────────┐
                     │  Client  │  (web / mobile)
                     └────┬─────┘
                          ▼
                  ┌───────────────┐
                  │  API Gateway  │  auth, rate-limit, routing
                  └───┬─────┬─────┘
          ┌───────────┘     └────────────┐
          ▼                              ▼
   ┌──────────────┐               ┌─────────────────┐
   │ Event Svc    │               │ Sharing/ACL Svc │
   │ (CRUD, RRULE)│               │ (who sees what) │
   └──────┬───────┘               └────────┬────────┘
          │                                │
          ▼                                ▼
       Postgres                         Postgres
       (events,                         (acls)
        rrules,
        exceptions)

   ┌──────────────┐          ┌─────────────────────┐
   │ Invite Svc   │          │ Reminder Scheduler  │
   │ (RSVPs)      │          │ (durable timer q)   │
   └──────┬───────┘          └──────────┬──────────┘
          │                             │
          ▼                             ▼
       Postgres                  Push/Email workers
```

### Why split into services?

- **Event service** owns event rows and recurrence rules. For recurring events we store the *rule*, not every future instance.
- **Sharing/ACL service** answers "can Bob see Alice's calendar?" — a hot path on every read.
- **Invite service** tracks RSVPs separately so invitation fan-out doesn't hold up event writes.
- **Reminder scheduler** is a classic durable timer queue. Firing 100k reminders/s at peak is a different problem than CRUD, so it runs on its own infrastructure. See `06-system-designs/reminder-alert`.


## The two ideas that define the whole design

Read these twice — everything else follows:

1. **Store the rule, not the occurrences.** An event that repeats every Monday for 10 years is **one row** with `rrule='FREQ=WEEKLY;BYDAY=MO'`, *not* 520 rows. When the client asks "what's in May 2026?" we *expand* the rule inside that window. This keeps storage small and updates cheap.

2. **UTC is the source of truth.** Store `starts_at` in UTC, store the user's IANA timezone (e.g. `America/Los_Angeles`) as a separate field, and convert only for display. "3pm in LA" is ambiguous around DST; "2026-03-08T15:00:00Z rendered in America/Los_Angeles" is not.

We'll make both concrete with running code in notebooks 2 and 3.


## What each of the next notebooks covers

- **Notebook 2 — Data Model & APIs**: Pydantic models for Event/Invitation/Room, HTTP endpoints, and a bad→best walkthrough of timezone storage (the most common beginner mistake).
- **Notebook 3 — Deep Dives**: recurrence expansion with real `dateutil.rrule`, overriding or cancelling a single occurrence, and a bad→best **free/busy** algorithm (naive O(n·m) vs. sweep-line merge).
